<a href="https://colab.research.google.com/github/rakasiwisurya/dfgai-lessons/blob/main/Latihan_Menggunakan_Open_Weight_untuk_Meringkas_Teks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Persiapan (Setup)

In [ ]:
# Menginstal library transformers dan lainnya
!pip install transformers[sentencepiece] datasets evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d0f7059fdd2f488e360b3d4326fbe17b8bdb3acbceb9c9e368e775f89d3aace0
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


# The Easy Way (Menggunakan Pipeline)

In [ ]:
!pip install "transformers==4.35.2" -q

In [ ]:
from transformers import pipeline, AutoTokenizer

# 1. Menyiapkan Teks Berita dalam Bahasa Indonesia
teks_indo = """
Kecerdasan Buatan (AI) telah mengalami perkembangan yang sangat pesat dalam beberapa tahun terakhir.
Teknologi ini mulai diterapkan di berbagai sektor, mulai dari kesehatan, pendidikan, hingga transportasi.
Salah satu terobosan terbesar adalah munculnya Large Language Models (LLM) yang mampu memproses dan menghasilkan teks layaknya manusia.
Meskipun memberikan banyak manfaat, seperti efisiensi kerja dan inovasi baru, perkembangan AI juga memunculkan kekhawatiran terkait etika dan masa depan pekerjaan.
Banyak ahli menyarankan perlunya regulasi yang ketat untuk memastikan AI digunakan demi kebaikan umat manusia, bukan sebaliknya.
Pemerintah Indonesia sendiri mulai merancang strategi nasional kecerdasan buatan untuk menghadapi era digital ini.
"""

MODEL_NAME = "cahya/bert2bert-indonesian-summarization"

# 2. Load Tokenizer Eksplisit (diperlukan untuk kompatibilitas model lama)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 3. Inisialisasi Pipeline
# Menggunakan "text2text-generation" karena "summarization" tidak tersedia di versi transformers terbaru
summarizer = pipeline("text2text-generation", model=MODEL_NAME, tokenizer=tokenizer)

# 4. Jalankan Peringkasan
hasil_pipeline = summarizer(teks_indo, max_new_tokens=12)

# Tampilkan Hasil
print("--- Hasil Ringkasan (Pipeline) ---")
print(hasil_pipeline[0]['generated_text'])

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your 

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/999M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1298: UserWarning: Unfeasible length constraints: `min_length` (20) is larger than the maximum possible length (13). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 13, its default value.
  warnings.warn(


--- Hasil Ringkasan (Pipeline) ---
perkembangan teknologi informasi memunculkan kekhawatiran terkait etika dan masa depan pekerjaan.


# The Engineer's Way (Pendekatan Manual)

## Memuat Komponen Secara Terpisah

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_checkpoint = "cahya/bert2bert-indonesian-summarization"

# 1. Memuat Tokenizer (Bertugas mengubah Teks <-> Angka)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 2. Memuat Model (Bertugas memproses Angka)
# Kita menggunakan AutoModelForSeq2SeqLM karena tugas kita adalah Sequence-to-Sequence (Teks panjang ke Teks pendek)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

print("Komponen berhasil dimuat secara manual!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Komponen berhasil dimuat secara manual!


## Proses Tokenisasi (Input)

In [ ]:
# Mengubah teks menjadi tensor (format angka PyTorch)
input_tokens = tokenizer(teks_indo, return_tensors="pt", padding="max_length", truncation=True, max_length=512)

# Mari kita lihat wujud aslinya
print("Bentuk Data (Shape):", input_tokens["input_ids"].shape)
print("5 Token Pertama (Angka):", input_tokens["input_ids"][0][:5])

Bentuk Data (Shape): torch.Size([1, 512])
5 Token Pertama (Angka): tensor([    3, 12925,  4994,    11, 10239])


## Generation dan Decoding Strategies (Output)

In [ ]:
# Menjalankan proses generasi (Inference)
summary_ids = model.generate(
    input_tokens["input_ids"],
    max_length=100,      # Batas maksimal panjang ringkasan
    min_length=20,       # Batas minimal ringkasan
    num_beams=5,         # Menggunakan 5 "berkas cahaya" (Beam Search) untuk mencari kalimat terbaik
    early_stopping=True, # Berhenti jika model merasa kalimat sudah lengkap
    no_repeat_ngram_size=2 # Mencegah model mengulang frasa yang sama (supaya tidak repetitif)
)

# Decoding: Mengubah angka hasil generasi kembali menjadi teks
ringkasan_manual = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("--- Hasil Ringkasan (Manual dengan Beam Search) ---")
print(ringkasan_manual)

--- Hasil Ringkasan (Manual dengan Beam Search) ---
perkembangan ai memunculkan kekhawatiran terkait etika dan masa depan pekerjaan. banyak ahli menyarankan perlunya regulasi yang ketat untuk memastikan ai digunakan demi kebaikan umat manusia.


## Evaluasi Model

In [ ]:
import evaluate

# 1. Menyiapkan Metrik
rouge = evaluate.load('rouge')

# 2. Menyiapkan Kunci Jawaban (Referensi Manusia)
# Anggaplah ini adalah ringkasan ideal yang dibuat oleh editor berita
referensi = ["Kecerdasan Buatan (AI) berkembang pesat dan membawa manfaat seperti efisiensi, namun memunculkan kekhawatiran etika sehingga perlu regulasi."]

# 3. Menghitung Skor
skor = rouge.compute(predictions=[ringkasan_manual], references=referensi)

print("Skor Evaluasi ROUGE:")
print(skor)

Skor Evaluasi ROUGE:
{'rouge1': np.float64(0.28571428571428564), 'rouge2': np.float64(0.05), 'rougeL': np.float64(0.2380952380952381), 'rougeLsum': np.float64(0.2380952380952381)}


# Tantangan Bahasa

## T5 (Text-to-Text Transfer Transformer)

In [ ]:
# 1. Ganti Checkpoint ke Model Bahasa Inggris (T5-Small)
model_eng_checkpoint = "t5-small"
tokenizer_eng = AutoTokenizer.from_pretrained(model_eng_checkpoint)
model_eng = AutoModelForSeq2SeqLM.from_pretrained(model_eng_checkpoint)

# 2. Teks Bahasa Inggris
text_eng = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
It is named after the engineer Gustave Eiffel, whose company designed and built the tower.
Locally nicknamed "La Dame de Fer" (French for "The Iron Lady"), it was constructed from 1887 to 1889
as the centerpiece of the 1889 World's Fair.
"""

# T5 memiliki keunikan: Kita harus memberi prefix "summarize: " di awal teks
input_text_formatted = "summarize: " + text_eng

# 3. Proses Manual (Tokenize -> Generate -> Decode)
inputs = tokenizer_eng(input_text_formatted, return_tensors="pt", max_length=512, truncation=True)

# Kita coba parameter kreatif: do_sample=True (Sampling) dengan temperature
summary_ids_eng = model_eng.generate(
    inputs["input_ids"],
    max_length=50,
    min_length=10,
    do_sample=True,   # Aktifkan sampling agar lebih variatif
    temperature=0.7   # Suhu kreativitas
)

summary_eng = tokenizer_eng.decode(summary_ids_eng[0], skip_special_tokens=True)

print("--- English Summary (T5 Model) ---")
print(summary_eng)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

--- English Summary (T5 Model) ---
the Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars. it was built from 1887 to 1889 as the centerpiece of the 1889 World's Fair.
